## What I'll look for

1. **VIP status and spending.** If the anomaly was planned, could wealthy passengers have been targeted? I'll look at total spending, destination, and VIP status. I'll also check whether passengers who shared a surname or cabin with a VIP were more likely to be transported.
2. **CryoSleep.** Were passengers in CryoSleep transported at a different rate than passengers who were awake?
3. **Age.** Did transport rates differ among children, teenagers, young adults, adults, and older passengers?
4. **Destination.** Were passengers headed to certain destinations more likely to be transported? I'll compare rates because busier destinations may have more transported passengers simply because more people were headed there.
5. **Home planet.** Did transport rates vary by home planet?
6. **Passenger ID.** Did transport rates differ between passengers with even and odd passenger IDs?
7. **Names.** I'll split full names into first names and surnames to look for possible family members who stayed in different cabins.
8. **Cabin.** I'll split the cabin field into deck, cabin number, and side. Then I'll compare transport rates across decks and between port and starboard.
9. **Onboard spending.** I'll check whether spending at the food court, spa, or VR deck was associated with being transported, looking at each service separately.
10. **Group travel.** Were passengers traveling in a group more likely to be transported than solo travelers?

These checks will help me look for patterns that might fit either a natural or a planned anomaly.

In [1]:
# Load and inspect the training data

# Importing the csv file into a pandas dataframe
import pandas as pd

df = pd.read_csv('train.csv')

#Creating a copy of the dataframe to avoid modifying the original dataframe
df_copy  = df.copy()

print(df_copy.head())

"""Bringing the missing values to the top of the dataframe to inspect them."""

# df_copy.sort_values(by="CryoSleep", na_position="first").to_csv("null_inspection.csv", index=False)


  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4         True  


'Bringing the missing values to the top of the dataframe to inspect them.'

In [2]:
# Calculate and summarize passenger spending

spending_columns = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
spending = df_copy[spending_columns]

# If one or two charges are missing and every recorded charge is zero,
# assume the missing charges are zero before calculating Total_Spend.
missing_count = spending.isna().sum(axis=1)
zero_count = spending.eq(0).sum(axis=1)
can_fill_spending = missing_count.isin([1, 2]) & (zero_count + missing_count).eq(len(spending_columns))
df_copy.loc[can_fill_spending, spending_columns] = spending.loc[can_fill_spending].fillna(0)

df_copy["Total_Spend"] = df_copy["RoomService"] + df_copy["FoodCourt"] + df_copy["ShoppingMall"] + df_copy["Spa"] + df_copy["VRDeck"]

# Checking to see what statics model can be used to fill in the missing values in the Total_Spend column
Total_spend = df_copy["Total_Spend"].sum()
print("Total Spend: ", Total_spend)

"""The average spend here might not be the best idea to 
fill in the missing values in the Total_Spend column because 
there are a lot of outliers in the data."""

print("Average Spend: ", Total_spend/len(df_copy))

Total Spend:  11557623.0
Average Spend:  1329.5321523064536


In [3]:
# Fill missing CryoSleep values and review what remains

# Restore the original values before applying the fill rules.
df_copy["CryoSleep"] = df["CryoSleep"].copy()

has_known_spending = df_copy[spending_columns].gt(0).any(axis=1)
has_zero_total = df_copy["Total_Spend"].eq(0)
missing_cryo_sleep = df_copy["CryoSleep"].isna()

df_copy.loc[missing_cryo_sleep & has_known_spending, "CryoSleep"] = False
df_copy.loc[missing_cryo_sleep & has_zero_total, "CryoSleep"] = True

print(df_copy["CryoSleep"].isnull().sum())

# There is something we can do here. We clearly see that some people were in cryo_sleep
# and if some people were in crying sleep showing the remaining data, 
# we can assume that they did not spend any money on the ship. So 
# we can fill in the missing values in the Total_Spend column with 0 for those who were in cryo_sleep.




# Exporting the modified dataframe to a new csv file
# df_copy.to_csv('temporary.csv', index=False)

0


In [4]:
df_copy.sort_values(by="CryoSleep", na_position="first").to_csv("null_inspection.csv", index=False)